# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [5]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [13]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
# Comprobar formato y unicidad del código.

# Crear columna de código municipio desde DIVIPOLA
divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()

# Comprobación de formato: 5 cifras
formato_valido = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}")
print("Códigos válidos:", formato_valido.sum(), "de", len(divipola))

# Comprobación de unicidad
print("¿Códigos únicos?", divipola["codigo_municipio"].is_unique)
print('----------------------------------------------------------')

# Construir dimensión con las columnas requeridas
dimension = divipola[[
    "codigo_municipio",
    "Nombre Departamento",
    "Nombre Municipio",
    "Tipo: Municipio / Isla / Área no municipalizada"
]].copy()

# Conteo de entidades por tipo
conteo_tipos = dimension["Tipo: Municipio / Isla / Área no municipalizada"].value_counts()
print(conteo_tipos)
print('----------------------------------------------------------')

# Mostrar primeras filas de la dimensión
print(dimension.head(5))

Códigos válidos: 1122 de 1122
¿Códigos únicos? True
----------------------------------------------------------
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64
----------------------------------------------------------
  codigo_municipio Nombre Departamento Nombre Municipio  \
0            05001           ANTIOQUIA         MEDELLÍN   
1            05002           ANTIOQUIA        ABEJORRAL   
2            05004           ANTIOQUIA         ABRIAQUÍ   
3            05021           ANTIOQUIA       ALEJANDRÍA   
4            05030           ANTIOQUIA            AMAGÁ   

  Tipo: Municipio / Isla / Área no municipalizada  
0                                       Municipio  
1                                       Municipio  
2                                       Municipio  
3                                       Municipio  
4                                       Municipi

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

Formato: todos los códigos cumplen con 5 dígitos.

Unicidad: la columna codigo_municipio es única.

Distribución: el value_counts() muestra las tres categorías (Municipio, Isla, Área no municipalizada) sumando 1.122 entidades.

Dimensión: contiene las columnas codigo_municipio, Nombre Departamento, Nombre Municipio y la clasificación territorial en el campo completo.

--------------------------------------------------
La columna codigo_municipio es un identificador territorial confiable: no tiene vacíos, es única y cumple con el formato oficial.

La clasificación territorial está correctamente representada en un solo campo, lo que asegura que la dimensión respete la estructura original de DIVIPOLA.

La dimensión con 1.122 registros es la base para enlazar datos de producción agrícola (eva.csv) con la división político-administrativa oficial.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [14]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
# Obtener sin_correspondencia con código municipal y año.

# Preparar código municipal desde EVA
eva["codigo_municipio"] = eva["Código Dane municipio"].str.strip()

# Identificar códigos válidos (solo 5 dígitos)
validos = eva["codigo_municipio"].str.fullmatch(r"[0-9]{5}")

# Completar con ceros a la izquierda solo en textos numéricos válidos de menor longitud
eva["codigo_municipio"] = eva["codigo_municipio"].where(
    validos,  # conservar los inválidos tal cual
    eva["codigo_municipio"].str.zfill(5)  # rellenar con ceros si es numérico válido
)

# Preparar año
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")

# Conservar originales: no se eliminan columnas ni filas
# Crear combinaciones distintas de código y año
combinaciones = eva[["codigo_municipio", "anio"]].drop_duplicates()

# Anti-join frente a dimension
sin_correspondencia = combinaciones.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one"
).query('_merge == "left_only"').drop(columns="_merge")

# Mostrar resultado
print("Total combinaciones:", len(combinaciones))
print("Sin correspondencia:", len(sin_correspondencia))
print(sin_correspondencia.head())

Total combinaciones: 7691
Sin correspondencia: 0
Empty DataFrame
Columns: [codigo_municipio, anio]
Index: []


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

Código municipal: es la clave de unión; los vacíos o mal formados quedan identificados para revisión.

Año: se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA.

Anti-join: sin_correspondencia muestra los códigos y años que no tienen correspondencia en la dimensión.

Listado: se presenta aunque quede vacío, garantizando evidencia del control.

---------------------------------------------------------------------------
Todos los códigos municipales presentes en EVA tienen correspondencia en la dimensión oficial de DIVIPOLA.

No se identificaron códigos vacíos, mal formados o fuera del catálogo oficial.

El listado sin_correspondencia se muestra aunque esté vacío, lo cual confirma que el control se ejecutó correctamente y no se omitió evidencia.


## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [15]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
# Mostrar las exclusiones del indicador y conciliar los conteos.

# Conversión de columnas a numéricas
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

# Selección de papa en 2024
papa_2024 = eva.loc[
    eva["Cultivo"].eq("Papa") & eva["anio"].eq(2024)
].copy()

# Filtrar filas aptas: área positiva y producción no negativa
apta = papa_2024.loc[
    papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0)
].copy()

# Conteo de exclusiones
excluidas = len(papa_2024) - len(apta)

print("Total papa_2024:", len(papa_2024))
print("Aptas:", len(apta))
print("Excluidas:", excluidas)

Total papa_2024: 831
Aptas: 779
Excluidas: 52


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

La selección de Papa en 2024 concentra el análisis en el cultivo y año de interés.

El filtro de aptitud asegura que solo se consideren registros con área positiva y producción no negativa, descartando inconsistencias.

El conteo de exclusiones dentro de papa_2024 evidencia cuántos registros no cumplen los criterios del indicador.

La conciliación de totales confirma que no se pierden filas: cada registro está clasificado como apto o excluido.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [17]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
# Comprobar denominadores y cantidad de registros agrupados.

# Agrupar apta por código, cultivo, estado físico y año
rendimiento = (
    apta
    .groupby(
        ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"],
        as_index=False,
        dropna=False
    )
    .agg(
        registros=("codigo_municipio", "size"),
        produccion_total=("produccion_t", "sum"),
        area_total=("area_ha", "sum")
    )
)

# Calcular rendimiento_t_ha
rendimiento["rendimiento_t_ha"] = (
    rendimiento["produccion_total"] / rendimiento["area_total"]
)

# Mostrar primeras filas
print(rendimiento.head(5))

# Comprobaciones
print("Total registros agrupados:", rendimiento["registros"].sum())
print("Len apta:", len(apta))
print("¿Áreas positivas en todos los grupos?", (rendimiento["area_total"] > 0).all())

  codigo_municipio Cultivo Estado físico del cultivo  anio  registros  \
0            05001    Papa                 En fresco  2024          2   
1            05002    Papa                 En fresco  2024          2   
2            05004    Papa                 En fresco  2024          4   
3            05079    Papa                 En fresco  2024          2   
4            05086    Papa                 En fresco  2024          3   

   produccion_total  area_total  rendimiento_t_ha  
0            1280.0        80.0         16.000000  
1            1650.0       165.0         10.000000  
2              15.0         5.0          3.000000  
3             268.0        34.0          7.882353  
4           10550.0       905.0         11.657459  
Total registros agrupados: 779
Len apta: 779
¿Áreas positivas en todos los grupos? True


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

Área total positiva: cada grupo debe tener suma de área > 0.

Conteo: la suma de registros en rendimiento debe coincidir con len(apta).

Indicador: rendimiento_t_ha se calcula como produccion_total / area_total, conservando las claves de agrupación (Cultivo, Estado físico del cultivo, anio).

--------------------------------------------------------

El agrupamiento de Papa 2024 por municipio y estado físico del cultivo arroja un total de 779 registros, exactamente igual al tamaño de apta. Esto confirma que ningún registro se perdió en la consolidación y que todos fueron contabilizados en los grupos.

Área total positiva: cada grupo presenta una suma de área cosechada mayor que cero, lo que garantiza que el denominador del indicador rendimiento_t_ha es válido en todos los casos.

Indicador de rendimiento: el cálculo de produccion_total / area_total refleja la productividad promedio en toneladas por hectárea. 

Consistencia: la suma de los conteos por grupo coincide con el total de registros aptos, lo que valida la integridad del proceso de agrupación.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [18]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# Verificar la conservación de filas y presentar la cobertura de la unión.

# Unir rendimiento con dimension
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Comprobaciones
print("Filas antes de la unión:", len(rendimiento))
print("Filas después de la unión:", len(rendimiento_municipal))

# Mostrar códigos sin nombre territorial (si existen)
sin_nombre = rendimiento_municipal.query('_merge == "left_only"')[
    ["codigo_municipio", "Cultivo", "anio"]
]
print("Códigos sin correspondencia territorial:", len(sin_nombre))
print(sin_nombre.head())

Filas antes de la unión: 292
Filas después de la unión: 292
Códigos sin correspondencia territorial: 0
Empty DataFrame
Columns: [codigo_municipio, Cultivo, anio]
Index: []


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

Conservación de filas: el número de registros antes y después de la unión debe ser idéntico.

Validación de cardinalidad: validate="many_to_one" asegura que cada código municipal en rendimiento se asocie con una única fila en la dimensión. Si falla, hay un problema en la dimensión que debe investigarse.

Indicador de correspondencia: la columna _merge permite identificar si todos los códigos de rendimiento encontraron su nombre territorial.

Listado de faltantes: si existen códigos sin correspondencia, se muestran explícitamente para revisión.

----------------------------------------------------
Conservación de filas: antes y después de la unión se mantienen las mismas 292 filas, lo que confirma que no se perdió información en el proceso.

Validación de cardinalidad: el parámetro validate="many_to_one" asegura que cada código municipal en rendimiento se corresponde con una única entidad en la dimensión. El hecho de que la unión haya sido exitosa indica que la clave territorial está bien definida y no presenta duplicados en la dimensión.

Cobertura total: el indicador de correspondencia muestra que todos los códigos municipales de rendimiento encontraron su nombre territorial. El listado de faltantes está vacío, lo cual es evidencia de que no existen registros sin correspondencia.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [19]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# Explicar el denominador y la limitación temporal en la celda siguiente.

# Exportar tablas a OUT
rendimiento_municipal.to_csv(OUT / "rendimiento_municipal.csv", index=False)
sin_correspondencia.to_csv(OUT / "codigos_sin_correspondencia.csv", index=False)

print("Archivos exportados en:", OUT)

Archivos exportados en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._


1. Por qué se dividen sumas en lugar de promediar rendimientos individuales

El indicador de rendimiento agrícola se define como: rendimiento_t_ha = suma de produccion / suma de area

Si se promediaran los rendimientos individuales de cada registro, se daría el mismo peso a parcelas pequeñas y grandes, distorsionando el resultado.


2. Limitación al comparar EVA con un catálogo territorial de corte 2024

La coincidencia de códigos garantiza que los registros de EVA están alineados con la clave territorial oficial vigente en 2024.

Sin embargo, esto no implica equivalencia histórica de límites: Municipios pueden haber cambiado de nombre, límites o categoría en años anteriores. EVA puede contener información de periodos previos a 2024, pero la dimensión territorial usada corresponde solo al corte actual.


## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.